# Coffee17 Physical Descriptor Audit V1

Development-only physical-descriptor audit for Coffee17.

This notebook performs **no neural training** and **does not materialize any fold's
locked test split**. It reconstructs the exact clean Coffee17 population and fold
manifests, then audits each fold using only that fold's train+val development images.

Frozen hypotheses:

1. Full Sour vs Partial Sour — CIELAB + warm-color coverage proxies.
2. Full Black vs Partial Black — CIELAB + luminance coverage curve.
3. Severe vs Slight Insect Damage — dark-component topology proxies.
4. Broken/Cut/Immature/Shell/Withered — explicit bean geometry.

Output:

`coffee17-physical-descriptor-audit-v1-analysis-package.zip`


In [ ]:
NOTEBOOK_BUILD = "COFFEE17-PHYSICAL-DESCRIPTOR-AUDIT-V1"
SCIENTIFIC_CODE_COMMIT = "3347667130da4452849e643561476f89126acfc0"
print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-physical-audit"
PROJECT = WORK / "coffee17-physical-descriptor-audit-v1"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

# ------------------------------------------------------------
# 1. Exact scientific code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

# Frozen unit tests for the descriptor extractor.
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/analysis/test_coffee17_physical_descriptors.py"
])

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

PROJECT.mkdir(parents=True, exist_ok=True)
AUDIT_ROOT = PROJECT / "fold_audits"
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
ANALYSIS = PROJECT / "analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 provenance and fold manifests.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_physical_audit_original.zip"
PROV = WORK / "coffee17_physical_audit_provenance"
CANONICAL = WORK / "coffee17_physical_audit_canonical"
FOLDS = WORK / "coffee17_physical_audit_folds"

for path in (PROV, CANONICAL, FOLDS):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV / "coffee17_provenance.json",
    FOLDS,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

print("clean_count:", fold_summary["clean_count"])
print("fold_counts:", json.dumps(fold_summary["fold_counts"], indent=2))

# ------------------------------------------------------------
# 3. Fold-specific development-only descriptor audits.
# ------------------------------------------------------------
print("\n=== 2/4 PHYSICAL DESCRIPTOR AUDIT ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_physical_audit_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    contract = materialize_preprocessing_development(
        CANONICAL,
        FOLDS / "clean_manifest.json",
        FOLDS / "fold_manifest.json",
        DEV,
        fold=fold,
    )
    assert contract["test_images_extracted"] is False
    assert not (DEV / "source/test").exists()

    fold_out = AUDIT_ROOT / f"fold_{fold}"
    shutil.rmtree(fold_out, ignore_errors=True)

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_coffee17_physical_descriptor_audit",
        "--dev-root", DEV,
        "--output-dir", fold_out,
        "--fold", str(fold),
    ], cwd=REPO)

    audit = json.loads((fold_out / "fold_audit.json").read_text(encoding="utf-8"))
    print(
        "samples=", audit["sample_count"],
        "mask_qc_fail=", audit["mask_qc_fail_count"],
        "outer_test_accessed=", audit["outer_test_accessed"],
        "training_executed=", audit["training_executed"],
    )

    shutil.rmtree(DEV, ignore_errors=True)

# ------------------------------------------------------------
# 4. Five-fold consistency summary.
# ------------------------------------------------------------
print("\n=== 3/4 FIVE-FOLD CONSISTENCY SUMMARY ===")

SUMMARY = ANALYSIS / "physical_descriptor_audit_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_coffee17_physical_descriptor_summary",
    "--audit-root", AUDIT_ROOT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))

def candidates(section):
    return [
        (name, row)
        for name, row in section["feature_consistency"].items()
        if row["exploratory_consistent_candidate"]
    ]

print("\nFULL vs PARTIAL SOUR candidates:")
for name, row in candidates(summary["full_vs_partial_sour"]):
    print(name, row)

print("\nFULL vs PARTIAL BLACK candidates:")
for name, row in candidates(summary["full_vs_partial_black"]):
    print(name, row)

print("\nSEVERE vs SLIGHT INSECT candidates:")
for name, row in candidates(summary["severe_vs_slight_insect"]):
    print(name, row)

print("\nSHAPE group consistently different:")
for name, row in summary["shape_family"]["kruskal_wallis_consistency"].items():
    if row["exploratory_consistent_group_difference"]:
        print(name, row)

print("\nOuter test accessed:", summary["outer_test_accessed"])
print("Training executed :", summary["training_executed"])

# ------------------------------------------------------------
# 5. Compact analysis package.
# ------------------------------------------------------------
print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")

PKG = WORK / "coffee17-physical-descriptor-audit-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(ANALYSIS, PKG / "analysis")
shutil.copytree(AUDIT_ROOT, PKG / "fold_audits")

manifest_dir = PKG / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)
for source in (
    PROV / "coffee17_provenance.json",
    PROV / "coffee17_raw_manifest.json",
    FOLDS / "clean_manifest.json",
    FOLDS / "fold_manifest.json",
    FOLDS / "fold_summary.json",
):
    shutil.copy2(source, manifest_dir / source.name)

protocol = REPO / "docs/protocols/COFFEE17_PHYSICAL_DESCRIPTOR_AUDIT_V1.md"
shutil.copy2(protocol, PKG / protocol.name)

metadata = {
    "notebook_build": NOTEBOOK_BUILD,
    "scientific_code_commit": SCIENTIFIC_CODE_COMMIT,
    "outer_test_materialized": False,
    "model_accessed": False,
    "training_executed": False,
}
(PKG / "PACKAGE_METADATA.json").write_text(
    json.dumps(metadata, indent=2) + "\n",
    encoding="utf-8",
)

zip_path = Path(
    shutil.make_archive(
        str(WORK / "coffee17-physical-descriptor-audit-v1-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("No locked test directory was materialized.")
print("No neural training was executed.")

for path in (REPO, PROV, CANONICAL, FOLDS, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
